In [0]:
from pyspark.sql import functions as F

# Lê a tabela tratada e seleciona os jogos classificados no período
partidas_analise = (
    spark.table("workspace.mvp_gramados.silver_partidas_gramados")
    .filter(F.col("status_gramado") == "dentro_do_periodo")
    .withColumn("ano", F.year("data"))
    .withColumn(
        "total_gols",
        F.col("mandante_Placar") + F.col("visitante_Placar")
    )
    .withColumn(
        "vitoria_mandante",
        F.when(
            F.col("mandante_Placar") > F.col("visitante_Placar"), 1
        ).otherwise(0)
    )
)

print("Partidas incluídas na análise:", partidas_analise.count())

display(
    partidas_analise.select(
        "data", "mandante", "visitante", "tipo_gramado",
        "mandante_Placar", "visitante_Placar",
        "total_gols", "vitoria_mandante"
    ).orderBy("data", "ID").limit(10)
)

In [0]:
# Agrupa as partidas por tipo de gramado
resumo_gramado = (
    partidas_analise
    .groupBy("tipo_gramado")
    .agg(
        F.count("*").alias("quantidade_partidas"),
        F.avg("total_gols").alias("media_gols"),
        F.sum("vitoria_mandante").alias("vitorias_mandante"),
        F.avg("vitoria_mandante").alias("taxa_vitoria_mandante")
    )
)

# Persiste o resultado na camada gold
resumo_gramado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.gold_resumo_gramado")

# Lê a tabela salva e formata os números para apresentação
display(spark.sql("""
    SELECT
        tipo_gramado,
        quantidade_partidas,
        ROUND(media_gols, 3) AS media_gols,
        vitorias_mandante,
        ROUND(taxa_vitoria_mandante * 100, 2)
            AS vitorias_mandante_pct
    FROM workspace.mvp_gramados.gold_resumo_gramado
    ORDER BY tipo_gramado
"""))

In [0]:
resumo_ano_gramado = (
    partidas_analise
    .groupBy("ano", "tipo_gramado")
    .agg(
        F.count("*").alias("quantidade_partidas"),
        F.avg("total_gols").alias("media_gols"),
        F.sum("vitoria_mandante").alias("vitorias_mandante"),
        F.avg("vitoria_mandante").alias("taxa_vitoria_mandante")
    )
)

resumo_ano_gramado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.gold_resumo_ano_gramado")

display(spark.sql("""
    SELECT
        ano,
        tipo_gramado,
        quantidade_partidas,
        ROUND(media_gols, 3) AS media_gols,
        vitorias_mandante,
        ROUND(taxa_vitoria_mandante * 100, 2)
            AS vitorias_mandante_pct
    FROM workspace.mvp_gramados.gold_resumo_ano_gramado
    ORDER BY ano, tipo_gramado
"""))

In [0]:
resumo_clube_gramado = (
    partidas_analise
    .groupBy("mandante", "tipo_gramado")
    .agg(
        F.count("*").alias("quantidade_partidas"),
        F.avg("total_gols").alias("media_gols"),
        F.sum("vitoria_mandante").alias("vitorias_mandante"),
        F.avg("vitoria_mandante").alias("taxa_vitoria_mandante")
    )
)

resumo_clube_gramado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.gold_resumo_clube_gramado")

display(spark.sql("""
    SELECT
        mandante,
        tipo_gramado,
        quantidade_partidas,
        ROUND(media_gols, 3) AS media_gols,
        vitorias_mandante,
        ROUND(taxa_vitoria_mandante * 100, 2)
            AS vitorias_mandante_pct
    FROM workspace.mvp_gramados.gold_resumo_clube_gramado
    ORDER BY tipo_gramado, quantidade_partidas DESC, mandante
"""))

In [0]:
spark.sql("""
    ALTER TABLE workspace.mvp_gramados.silver_partidas_gramados
    ALTER COLUMN arena COMMENT
    'Nome do estádio conforme a base de partidas. Utilizado para associar a partida à pesquisa de gramados. Nomes diferentes podem representar o mesmo estádio.'
""")

display(spark.sql("""
    DESCRIBE TABLE workspace.mvp_gramados.silver_partidas_gramados
"""))

In [0]:
tabela = "workspace.mvp_gramados.silver_partidas_gramados"

comentarios = {
    "ID": "Identificador da partida na fonte original. Sem valores ausentes ou repetidos no recorte analisado.",
    "rodada": "Número da rodada do campeonato. Coluna originalmente chamada rodata.",
    "data": "Data da partida, utilizada no recorte de 2023–2024 e na verificação da validade do gramado.",
    "hora": "Horário informado na fonte original, mantido como texto. Fuso horário não validado.",
    "mandante": "Clube que possui o mando da partida, independentemente do estádio utilizado.",
    "visitante": "Clube visitante da partida.",
    "formacao_mandante": "Formação tática do mandante informada na fonte. Não utilizada nos indicadores deste MVP.",
    "formacao_visitante": "Formação tática do visitante informada na fonte. Não utilizada nos indicadores deste MVP.",
    "tecnico_mandante": "Nome do técnico do mandante informado na fonte.",
    "tecnico_visitante": "Nome do técnico do visitante informado na fonte.",
    "vencedor": "Vencedor informado na fonte; o hífen indica empate. Os indicadores do MVP são calculados pelos placares.",
    "mandante_Placar": "Quantidade de gols marcados pelo mandante na partida.",
    "visitante_Placar": "Quantidade de gols marcados pelo visitante na partida.",
    "mandante_Estado": "Sigla do estado do clube mandante conforme a fonte.",
    "visitante_Estado": "Sigla do estado do clube visitante conforme a fonte.",
    "tipo_gramado": "Classificação da pesquisa: natural, sintetico, hibrido ou nao_confirmado. A aplicação à partida depende do período de validade.",
    "inicio_validade": "Início inclusivo do intervalo considerado para a classificação nesta pesquisa. Não representa necessariamente a data de instalação.",
    "fim_validade": "Fim inclusivo do intervalo considerado para a classificação nesta pesquisa. Não representa necessariamente a data de retirada ou troca.",
    "fonte_url": "Endereço ou endereços das fontes consultadas na pesquisa sobre o gramado.",
    "observacoes": "Notas sobre evidências, nomes alternativos do estádio, limitações e hipóteses de continuidade da classificação.",
    "status_gramado": "Resultado da verificação: sem_classificacao, pesquisa_pendente, periodo_nao_informado, dentro_do_periodo ou fora_do_periodo. Apenas dentro_do_periodo entra nas análises Gold."
}

for coluna, descricao in comentarios.items():
    spark.sql(
        f"ALTER TABLE {tabela} "
        f"ALTER COLUMN `{coluna}` COMMENT '{descricao}'"
    )

display(spark.sql(f"DESCRIBE TABLE {tabela}"))

In [0]:
descricoes_tabelas = {
    "bronze_partidas":
        "Dados brutos do arquivo campeonato-brasileiro-full.csv. "
        "Uma linha por registro da fonte, com todas as colunas carregadas como texto.",

    "bronze_gramados":
        "Dados brutos do arquivo gramados.csv, produzido pela pesquisa de gramados. "
        "Inclui classificação, intervalo considerado, fontes e observações.",

    "silver_gramados":
        "Pesquisa de gramados com datas convertidas. "
        "Nesta versão, há um registro por nome de estádio utilizado na associação. "
        "Nomes diferentes podem representar o mesmo estádio. "
        "Limitações e hipóteses da pesquisa estão nas observações.",

    "silver_partidas_gramados":
        "Partidas de 2023 e 2024 associadas à pesquisa pelo nome exato do estádio. "
        "Uma linha por partida, incluindo registros com pesquisa pendente. "
        "O campo status_gramado verifica a classificação e seu período de validade.",

    "gold_resumo_gramado":
        "Indicadores das partidas com status dentro_do_periodo. "
        "Uma linha por tipo de gramado, com quantidade de partidas, média do total "
        "de gols, vitórias do mandante e taxa de vitória entre 0 e 1.",

    "gold_resumo_ano_gramado":
        "Indicadores das partidas com status dentro_do_periodo. "
        "Uma linha por combinação de ano e tipo de gramado. "
        "A taxa de vitória do mandante é armazenada entre 0 e 1.",

    "gold_resumo_clube_gramado":
        "Indicadores das partidas com status dentro_do_periodo. "
        "Uma linha por combinação de clube mandante e tipo de gramado. "
        "A taxa de vitória do mandante é armazenada entre 0 e 1. "
        "As comparações são descritivas e não isolam o efeito do gramado."
}

for tabela, descricao in descricoes_tabelas.items():
    spark.sql(
        f"COMMENT ON TABLE workspace.mvp_gramados.{tabela} "
        f"IS '{descricao}'"
    )

print("Descrições das sete tabelas registradas.")

In [0]:
display(spark.sql("""
    SELECT
        table_name,
        comment
    FROM workspace.information_schema.tables
    WHERE table_schema = 'mvp_gramados'
    ORDER BY table_name
"""))